# 通用近似器 / Universal Approximator

## 通用近似定理 / Universal Approximation Theorem

1989 年 Cybenko（sigmoid 激活）与 1991 年 Hornik（更广的激活族）分别证明了：具有 **单层足够宽** 的隐藏层 + 非线性激活的前馈神经网络，可以在紧致集上 **一致逼近** 任意连续函数 $f: \mathbb{R}^n \to \mathbb{R}^m$。

Cybenko (1989, for sigmoid activations) and Hornik (1991, for a broader class) independently proved that a feedforward neural network with **one sufficiently wide** hidden layer and a non-linear activation can **uniformly approximate** any continuous function $f: \mathbb{R}^n \to \mathbb{R}^m$ on a compact set.

这一定理回答了"神经网络表示能力有多强"的问题，但 **没有** 回答"多宽够用 / 如何训练"——后者才是实际工程的难点。本章用 **纯 NumPy** 实现一个两隐藏层全连接网络，从零开始演示前向 / 反向传播 / 梯度下降，让你在没有深度学习框架的情况下理解神经网络内部到底在做什么。

The theorem settles the question of *representation power*, but says **nothing** about *how wide is wide enough* or *how to train in practice* — those are the real engineering challenges. This chapter implements a two-hidden-layer fully-connected network entirely in **NumPy**, walking through forward-prop, back-prop, and gradient descent from scratch, so you can understand what a neural network is actually doing without any deep-learning framework in the way.

<!-- bilingual -->

In [ ]:
"""
Universal Function Approximators 函数拟合器
Multi-layer Perceptrons (NN) is Universal Approximators provided sufficiently broad and deep
http://deeplearning.cs.cmu.edu/F20/index.html
"""

import numpy as np

## 全连接层 / Fully-Connected Layer

一个全连接层把 $n_{\text{in}}$ 维输入向量 $\mathbf{x}$ 映射到 $n_{\text{out}}$ 维输出向量：

$$\mathbf{y} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$$

其中 $\mathbf{W}\in\mathbb{R}^{n_{\text{out}}\times n_{\text{in}}}$ 是权重矩阵，$\mathbf{b}\in\mathbb{R}^{n_{\text{out}}}$ 是偏置，$\sigma(\cdot)$ 是逐元素激活函数（本实现支持 sigmoid / tanh / ReLU / linear）。反向传播时需要缓存三样东西：**输入 $\mathbf{x}$**（算 $\partial \mathbf{y}/\partial \mathbf{W}$ 用）、**预激活 $\mathbf{z} = \mathbf{Wx}+\mathbf{b}$**（算 $\sigma'(\mathbf{z})$ 用）、**输出 $\mathbf{y}$**。权重用小幅度的均匀随机数初始化以打破对称。

A fully-connected layer maps an $n_{\text{in}}$-dim input $\mathbf{x}$ to an $n_{\text{out}}$-dim output by

$$\mathbf{y} = \sigma(\mathbf{W}\mathbf{x} + \mathbf{b})$$

where $\mathbf{W}\in\mathbb{R}^{n_{\text{out}}\times n_{\text{in}}}$, $\mathbf{b}\in\mathbb{R}^{n_{\text{out}}}$, and $\sigma$ is an element-wise activation (the implementation supports sigmoid / tanh / ReLU / linear). Back-propagation needs three cached quantities: the **input $\mathbf{x}$** (for $\partial \mathbf{y}/\partial \mathbf{W}$), the **pre-activation $\mathbf{z}=\mathbf{Wx}+\mathbf{b}$** (for $\sigma'(\mathbf{z})$), and the **output $\mathbf{y}$**. Weights are initialised with small random values to break symmetry.

<!-- bilingual -->

In [ ]:
class Layer:
    """ A Fully Connected Layer """
    def __init__(self, n_input, n_output, activation=None, weights=None, bias=None):
        """
        :param int n_input: Number of input nodes/neurons of previous layer
        :param int n_output: Number of output nodes/neurons of this layer
        :param str activation: Type of activation function
        :param weights : Weight of input connections
        :param bias : Bias of input connections
        """
        self.weights = weights if weights is not None else np.random.randn(n_input, n_output) / np.sqrt(n_input) * 2 # Normalization
        self.bias = bias if bias is not None else np.random.rand(n_output) * 0.2
        self.weights_update = np.zeros_like(self.weights)  # weights_new = weights_old + weights_update
        self.bias_update = np.zeros_like(self.bias)  # bias_new = bias_old + bias_update
        self.activation = activation  # relu tanh or sigmoid
        self.activation_output = None  # Output/activation value of this layer
        self.error = None  # Error or discrepancy of output/activation value
        self.delta = None  # Delta of X@W + b, delta = error*activation_derivative(output)
    def activate(self, X):
        # Forward propagation function
        r = np.dot(X, self.weights) + self.bias # X@W + b
        # Output of fully connected layer, o (activation_output)
        self.activation_output = self._apply_activation(r)
        return self.activation_output
    def _apply_activation(self, r):
        if self.activation is None:
            return r
        elif self.activation == 'relu':
            return np.maximum(r, 0)
        elif self.activation == 'tanh':
            return np.tanh(r)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-r))
        return r
    def apply_activation_derivative(self, y):
        # Calculate the derivative of activation function
        if self.activation is None: # No activation function, derivative is 1
            return np.ones_like(y)
        # ReLU
        elif self.activation == 'relu':
            grad = np.array(y, copy=True)
            grad[y > 0] = 1.
            grad[y <= 0] = 0.
            return grad
        # Tanh
        elif self.activation == 'tanh':
            return 1 - y ** 2
        # Sigmoid
        elif self.activation == 'sigmoid':
            return y * (1 - y)
        return y

## 前向传播与反向传播 / Forward and Back Propagation

**前向传播** 就是逐层套用 `Layer.activate`：$\mathbf{x}\to\mathbf{y}_1\to\mathbf{y}_2\to\cdots\to\mathbf{y}_L$。

**反向传播** 用链式法则把损失对参数的梯度从输出层逐层往回传。对 MSE 损失 $\mathcal{L}=\tfrac12 \|\hat{\mathbf{y}}-\mathbf{y}\|^2$，输出层误差 $\boldsymbol{\delta}_L = (\hat{\mathbf{y}}-\mathbf{y}) \odot \sigma'_L(\mathbf{z}_L)$；隐藏层误差逐层回传：

$$\boldsymbol{\delta}_\ell = (\mathbf{W}_{\ell+1}^{\top}\boldsymbol{\delta}_{\ell+1}) \odot \sigma'_\ell(\mathbf{z}_\ell)$$

权重与偏置的更新规则：

$$\mathbf{W}_\ell \leftarrow \mathbf{W}_\ell - \eta\,\boldsymbol{\delta}_\ell\,\mathbf{x}_\ell^{\top},\quad \mathbf{b}_\ell \leftarrow \mathbf{b}_\ell - \eta\,\boldsymbol{\delta}_\ell$$

其中 $\eta$ 是学习率。下面的 `NeuralNetwork.backpropagation` 就是这组公式的直译。

**Forward prop** is simply stacking `Layer.activate` calls: $\mathbf{x}\to\mathbf{y}_1\to\cdots\to\mathbf{y}_L$.

**Back prop** uses the chain rule to propagate the loss gradient from the output back through every layer. For MSE loss $\mathcal{L}=\tfrac12 \|\hat{\mathbf{y}}-\mathbf{y}\|^2$, the output-layer error is $\boldsymbol{\delta}_L = (\hat{\mathbf{y}}-\mathbf{y}) \odot \sigma'_L(\mathbf{z}_L)$; hidden-layer errors propagate backwards:

$$\boldsymbol{\delta}_\ell = (\mathbf{W}_{\ell+1}^{\top}\boldsymbol{\delta}_{\ell+1}) \odot \sigma'_\ell(\mathbf{z}_\ell)$$

with weight and bias updates

$$\mathbf{W}_\ell \leftarrow \mathbf{W}_\ell - \eta\,\boldsymbol{\delta}_\ell\,\mathbf{x}_\ell^{\top},\quad \mathbf{b}_\ell \leftarrow \mathbf{b}_\ell - \eta\,\boldsymbol{\delta}_\ell$$

where $\eta$ is the learning rate. The `NeuralNetwork.backpropagation` method below is a direct translation of these equations.

<!-- bilingual -->

In [ ]:
class NeuralNetwork:
    def __init__(self):
        self._layers = []
    def add_layer(self, layer):
        self._layers.append(layer)
    def feed_forward(self, X):
        # Forward propagation
        for layer in self._layers:
            X = layer.activate(X)
        return X
    def backpropagation(self, X, y, learning_rate):
        # Calculate delta of each layer
        output = self.feed_forward(X)
        for i in reversed(range(len(self._layers))): # reverse looping
            layer = self._layers[i]
            if layer == self._layers[-1]: # output layer
                layer.error = y - output
                # calculate delta of final layer
                layer.delta = layer.error * layer.apply_activation_derivative(output)
            else: # hidden layer
                next_layer = self._layers[i + 1]
                layer.error = np.dot(next_layer.weights, next_layer.delta)
                layer.delta = layer.error*layer.apply_activation_derivative(layer.activation_output)
        # Calculate weight_update and bias_update
        for i in range(len(self._layers)):
            layer = self._layers[i]
            # o_i is the output/activation value of previous layer
            o_i = np.atleast_2d(X if i == 0 else self._layers[i - 1].activation_output)
            layer.weights_update += layer.delta * o_i.T * learning_rate
            layer.bias_update += layer.delta * learning_rate
    def train(self, X_train, X_test, y_train, y_test, learning_rate, max_epochs, batch_size):
        mses = []  # Mean square errors
        for i in range(max_epochs):
            for j in range(len(X_train)):  # one sample each train
                self.backpropagation(X_train[j], y_train[j], learning_rate)
                if j % batch_size == batch_size - 1:  # averaging over this batch
                    for k in range(len(self._layers)):
                        layer = self._layers[k]
                        layer.weights += layer.weights_update / batch_size
                        layer.bias += layer.bias_update / batch_size
                        layer.weights_update = np.zeros_like(layer.weights)  # resatrt averaging over this batch
                        layer.bias_update = np.zeros_like(layer.bias)
            if i % 10 == 0:
                # print MSE Loss
                mse = np.mean(np.square(y_train - self.feed_forward(X_train)))
                mse_test = np.mean(np.square(y_test - self.feed_forward(X_test)))
                mses.append(mse)
                print('Epoch: #%s, Train MSE: %f, Test MSE: %f' %(i, float(mse), float(mse_test)))
        return mses

## 目标函数与训练数据 / Target Function and Training Data

我们选一个非线性、光滑、二维到一维的函数 $f(x_0, x_1) = \sin(x_0) + \sin(x_1)$ 作为 benchmark。它同时包含两个正弦周期，适合展示 MLP 拟合周期性特征的能力，且避免了一维问题过于简单的风险。训练集 200 样本、测试集 20 样本，$\mathbf{x}$ 均匀采样于 $[0, 5]^2$；训练标签人为加入幅度 $[0, 2]$ 的均匀噪声，用以检验网络的抗噪能力。

We use a non-linear, smooth, 2-D-to-1-D function $f(x_0, x_1) = \sin(x_0) + \sin(x_1)$ as the benchmark. It contains two sinusoidal modes at once, which nicely showcases an MLP's ability to fit periodic features while avoiding the over-simplicity of a 1-D toy problem. The training set has 200 samples and the test set 20, with $\mathbf{x}$ sampled uniformly from $[0, 5]^2$. Uniform noise in $[0, 2]$ is added to the training labels to check the network's noise robustness.

<!-- bilingual -->

In [ ]:
def f(X):
    """
    y =  sin(x0) + sin(x1)
    """
    return np.sum(np.sin(X), axis = 1)

In [ ]:
num_train = 200
num_test = 20
X_train = np.random.rand(num_train,2) * 5  # x1 x2 [0-5]
y_train = f(X_train).reshape(num_train,1) + 2 * np.random.rand(num_train,1)
X_test = np.random.rand(num_test,2) * 5
y_test = f(X_test).reshape(num_test,1)

## 网络架构与超参数 / Network Architecture and Hyperparameters

架构：**2 → 20 → 20 → 1**，两个 20 宽的 sigmoid 隐藏层，线性输出层（回归任务不需要输出激活）。sigmoid 容易产生梯度消失，所以网络不能做得太深——本例仅两个隐藏层。

- **学习率** $\eta=0.1$：MSE + sigmoid 组合下一个经验上合适的值；更大会震荡，更小收敛慢。
- **max_epochs=201**：第一轮先看收敛趋势，若未收敛再手动跑一次续训（见下一个代码 cell）。
- **batch_size=1**（即 SGD）：最简单的在线学习，每个样本更新一次。代价是收敛抖动大、速度慢；真实场景通常会用 mini-batch。

Architecture: **2 → 20 → 20 → 1**, with two 20-wide sigmoid hidden layers and a linear output (regression doesn't need an output activation). Sigmoid tends to cause vanishing gradients, so the network cannot be deep — two hidden layers is our cap here.

- **Learning rate** $\eta=0.1$: an empirically reasonable choice for MSE + sigmoid; larger values oscillate, smaller ones converge too slowly.
- **max_epochs=201**: a first pass to watch the convergence trend; if not converged yet, manually re-train for another round (see the next code cell).
- **batch_size=1** (plain SGD): the simplest online learning, one update per sample. The trade-off is noisy updates and slow convergence; real-world code would use mini-batches.

<!-- bilingual -->

In [ ]:
nn = NeuralNetwork()  # 3 layers, 2 inputs, 1 outputs
nn.add_layer(Layer(2, 20 , 'sigmoid'))  # hidden layer 1, 2 input => 20 output
nn.add_layer(Layer(20, 20, 'sigmoid'))  # hidden layer 2, 20 => 20
nn.add_layer(Layer(20, 1))  # output layer, 20 => 1, no activation

mses = nn.train(X_train, X_test, y_train, y_test, learning_rate=0.1, max_epochs=201, batch_size=1)

In [ ]:
mses = nn.train(X_train, X_test, y_train, y_test, learning_rate=0.1, max_epochs=201, batch_size=1)

## 结果可视化与结论 / Visualising the Result

用线框图同时画出 **真实曲面** $y = \sin(x_0)+\sin(x_1)$ 和 **网络预测曲面** $\hat{y}(x_0, x_1)$。网络只在 $[0, 5]^2$ 区域内见过训练样本，但我们在 $[0, 8]^2$ 上做预测——超出训练区域的部分是 **外推**，sigmoid 网络在外推上表现通常不好（激活容易饱和到常数）。观察这个失效模式比观察拟合好坏更有教学价值：它提醒我们神经网络并不"理解"函数本身，它只是在训练数据的分布范围内做了高维插值。

We render both the **true surface** $y = \sin(x_0)+\sin(x_1)$ and the **network prediction** $\hat{y}(x_0, x_1)$ as wireframes. The network only saw training samples inside $[0, 5]^2$, but we predict over $[0, 8]^2$ — everything outside $[0, 5]^2$ is **extrapolation**, where sigmoid networks typically perform poorly (activations saturate to constants). Paying attention to this failure mode is more informative than admiring the fit: it reminds us that a neural net does not "understand" the function — it is just high-dimensional interpolation within the training distribution.

<!-- bilingual -->

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

n = 500
x0, x1 = np.meshgrid(np.linspace(0, 8, n), np.linspace(0, 8, n))
y = np.sin(x0) + np.sin(x1)
X_new = np.c_[x0.ravel(), x1.ravel()]
y_predict = nn.feed_forward(X_new).reshape(n,n)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': '3d'})
ax.plot_wireframe(x0, x1, y, rstride=20,
                  cstride=20, linewidth=0.5,
                  color='blue')
ax.plot_wireframe(x0, x1, y_predict, rstride=20,
                  cstride=20, linewidth=0.5,
                  color='orangered')
ax.plot_surface(x0, x1, y_predict-y-2, cmap=plt.get_cmap('rainbow'))